# DE Overview

In [ ]:
%autosave 0

In [ ]:
import os
import sys
from pathlib import Path

# Set the base project directory
base_proj_dir = Path("/projects/site/pred/ihb-g-deco/USERS/adaml9/intestine_fate/notebooks/tf_ko_panel_analysis")

# Append necessary paths for module imports
sys.path.append("/projects/site/pred/ihb-g-deco/USERS/adaml9/bioinfo-blueprint/src/python")
sys.path.append(str(base_proj_dir))

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import delnx as dx
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from dynaconf import Dynaconf
from tqdm import tqdm
from functools import reduce
import numpy as np
import pandas as pd
import marsilea as ma 
import marsilea.plotter as mp 
import scipy.cluster.hierarchy as sch

# Load settings
settings = Dynaconf(
    settings_files=[base_proj_dir / 'settings.toml', base_proj_dir / '.secrets.toml'],
)

sc.settings.figdir = settings.ANALYSIS.figures_dir

In [ ]:
data_dir = Path(settings.IO.combined_data) / "all-sample" / "DGE_filtered"
anndata_dir = Path(settings.IO.anndata)

## Process DE results

### negbinom results

In [ ]:
# Load DE results (clustering based annotations, neg binom regression on counts)
de_results = pd.read_csv("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/tf_ko_screen/panel/tables/annotation_de_results_less_stringent/tf_ko_panel_contrastiveVI_de_results_annotations.tsv", sep="\t")

# Rename cell_type into group and condition into test_condition
de_results = de_results.rename(columns={"annotation": "group"})

# Escape groups
de_results["group"] = de_results["group"].str.replace(" ", "_").str.replace("/", "_") 

In [ ]:
coef_col = "coef"
pval_col = "padj"
pval_thresh = 0.05
coef_thresh = 0.5

sig_mask = (de_results[pval_col] < pval_thresh)
up_mask = sig_mask & (de_results[coef_col] > coef_thresh)
down_mask = sig_mask & (de_results[coef_col] < -coef_thresh)

de_results["significant"] = "NS"
de_results.loc[up_mask, "significant"] = "Up"
de_results.loc[down_mask, "significant"] = "Down"

### pyDESEQ2 results

In [ ]:
# Load DE results
de_results = pd.read_csv("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/tf_ko_screen/panel/tables/annotation_de_results_less_stringent/tf_ko_panel_contrastiveVI_de_results_annotations_pb.tsv", sep="\t")

In [ ]:
de_results.head()

In [ ]:
# Load DE results
de_results = pd.read_csv("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/tf_ko_screen/panel/tables/annotation_de_results/tf_ko_panel_contrastiveVI_de_results_annotations_pb.tsv", sep="\t")

# Rename cell_type into group and condition into test_condition
de_results = de_results.rename(columns={"cell_type": "group", "condition": "test_condition", "gene_name": "feature"})

In [ ]:
base_col = "baseMean"
coef_col = "log2FoldChange"
pval_col = "padj"
pval_thresh = 0.05
coef_thresh = 0.5
base_thresh = 1.0

sig_mask = (de_results[pval_col] < pval_thresh) & (de_results[base_col] > base_thresh)
up_mask = sig_mask & (de_results[coef_col] > coef_thresh)
down_mask = sig_mask & (de_results[coef_col] < -coef_thresh)

de_results["significant"] = "NS"
de_results.loc[up_mask, "significant"] = "Up"
de_results.loc[down_mask, "significant"] = "Down"

### Make sure that only relevant features are present

In [ ]:
# Remove genes no in keep_genes
keep_genes = de_results["feature"].unique().tolist()

# Get filtered gene list
with open(anndata_dir / "tf_ko_panel_contrastiveVI_raw_annotated_filtered_genes.txt", "r") as f:
    filter_genes = [line.strip() for line in f.readlines()]
    
# Intersect
keep_genes = set(keep_genes).intersection(filter_genes)

# Subset
de_results = de_results[de_results["feature"].isin(keep_genes)]

#### Plot number of DE genes

In [ ]:
n_des_df = de_results.query("significant != 'NS'").groupby(["test_condition", "group", "significant"]).size().unstack(fill_value=0).reset_index()

In [ ]:
n_des_df.query('test_condition == "Lmx1b"')

In [ ]:
# Export n_des_df table
#n_des_df.to_csv(Path(settings.ANALYSIS.tables_dir) / "tf_ko_panel_contrastiveVI_de_summary.tsv", sep="\t", index=False)

In [ ]:
group_order = ["Stem_Cells", "TA_Cells", "Cycling_TA_Cells", "Enterocytes", "Goblet_Cells", "Paneth_Cells", "EEC_Progenitors", "EC_Cells", "X_Cells", "I_N_Cells", "K_Cells", "D_Cells"]

# Get dataframe for plotting
df = n_des_df.copy()
df["group"] = df["group"].astype(pd.CategoricalDtype(categories=group_order, ordered=True))
df["label"] = df["test_condition"].astype(str) + " | " + df["group"].astype(str)

# Melt 
plot_df = df.melt(
    id_vars=["test_condition", "group", "label"],
    value_vars=["Down", "Up"],
    var_name="Direction", value_name="Count"
)
plot_df.loc[plot_df["Direction"] == "Down", "Count"] *= -1

# find max Up per (group, condition) pair
max_up = df.groupby(["group", "test_condition"])["Up"].max().reset_index()

# sort by group, then descending Up
max_up = max_up.sort_values(["group", "Up"], ascending=[True, False])
# new label order
label_order = (max_up["test_condition"].astype(str) + " | " + max_up["group"].astype(str)).tolist()
plot_df["label"] = pd.Categorical(plot_df["label"], categories=label_order, ordered=True)
plot_df = plot_df.sort_values(by="label")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Colors
cmap = {
    # proliferative axis
    "Stem_Cells": "#1f77b4",          # deep blue
    "TA_Cells": "#4c91c6",            # medium blue
    "Cycling_TA_Cells": "#7aaed6",    # light blue

    # absorptive lineage
    "Enterocytes": "#2ca02c",         # green

    # secretory lineage
    "Goblet_Cells": "#ff7f0e",        # orange
    "Paneth_Cells": "#ffbb78",        # light orange

    # eec progenitors
    "EEC_Progenitors": "#9467bd",  # purple

    # eec subtypes
    "EC_Cells": "#d62728",           # strong red
    "X_Cells": "#e377c2",            # magenta
    "D_Cells": "#17becf",            # cyan
    "K_Cells": "#bcbd22",            # olive
    "I_N_Cells": "#7f7f7f"           # dark grey
}

group_colors = {g: cmap[g] for g in group_order}


summary_rows = []

for lbl in plot_df["label"].unique():
    row = plot_df[plot_df["label"] == lbl]

    up = row.loc[row["Direction"] == "Up", "Count"].sum()
    down = row.loc[row["Direction"] == "Down", "Count"].sum()
    total = up + abs(down)
    group = row["group"].iloc[0]

    summary_rows.append({
        "label": lbl,
        "group": group,
        "up": up,
        "down": down,
        "total": total
    })

summary_df = pd.DataFrame(summary_rows)


summary_df["group_order"] = summary_df["group"].map(
    {g: i for i, g in enumerate(group_order)}
)

summary_df = (
    summary_df
    .sort_values(["group_order", "total"], ascending=[True, False])
    .reset_index(drop=True)
)

# Rebuild plotting arrays
labels = summary_df["label"].tolist()
x = np.arange(len(labels))
counts = list(zip(summary_df["up"], summary_df["down"]))
colors = [group_colors[g] for g in summary_df["group"]]

fig, ax = plt.subplots(figsize=(20, 4))

for xi, (up, down), col, lbl in zip(x, counts, colors, labels):
    total = up + abs(down)
    ax.bar(xi, total, color=col, edgecolor="none")


ax.axhline(0, color="black", linewidth=1)
ax.set_xlim(-0.5, len(labels) - 0.5)
ax.set_ylabel("Number of DE features")

plt.tight_layout()
plt.grid(False)
#plt.savefig(
#    sc.settings.figdir / "tf_ko_panel_de_results_summary_barplot.pdf",
#    dpi=300
#)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Colors
cmap = {
    # proliferative axis
    "Stem_Cells": "#1f77b4",          # deep blue
    "TA_Cells": "#4c91c6",            # medium blue
    "Cycling_TA_Cells": "#7aaed6",    # light blue

    # absorptive lineage
    "Enterocytes": "#2ca02c",         # green

    # secretory lineage
    "Goblet_Cells": "#ff7f0e",        # orange
    "Paneth_Cells": "#ffbb78",        # light orange

    # eec progenitors
    "EEC_Progenitors": "#9467bd",  # purple

    # eec subtypes
    "EC_Cells": "#d62728",           # strong red
    "X_Cells": "#e377c2",            # magenta
    "D_Cells": "#17becf",            # cyan
    "K_Cells": "#bcbd22",            # olive
    "I_N_Cells": "#7f7f7f"           # dark grey
}

group_colors = {g: cmap[g] for g in group_order}

# Compute summary table
summary_rows = []

for lbl in plot_df["label"].unique():
    row = plot_df[plot_df["label"] == lbl]

    up = row.loc[row["Direction"] == "Up", "Count"].sum()
    down = row.loc[row["Direction"] == "Down", "Count"].sum()
    total = up + abs(down)
    group = row["group"].iloc[0]

    summary_rows.append({
        "label": lbl,
        "group": group,
        "up": up,
        "down": down,
        "total": total
    })

summary_df = pd.DataFrame(summary_rows)
summary_df["condition"] = summary_df["label"].str.split(" | ").str[0]

# Add group ordering for sorting
summary_df["group_order"] = summary_df["group"].map(
    {g: i for i, g in enumerate(group_order)}
)

summary_df = (
    summary_df
    .sort_values(["group_order", "total"], ascending=[True, False])
    .reset_index(drop=True)
)

# ------ KEY PART: select top 5 per group ------
top5_df = (
    summary_df
    .assign(rank=lambda d: d.groupby("group")["total"].rank(method="first", ascending=False))
    .query("rank <= 5")
)

# Now use top5_df directly for plotting
labels = top5_df["label"].tolist()
x = np.arange(len(labels))
counts = list(zip(top5_df["up"], top5_df["down"]))
colors = [group_colors[g] for g in top5_df["group"]]

fig, ax = plt.subplots(figsize=(20, 4))

for xi, (up, down), col, lbl in zip(x, counts, colors, labels):
    total = up + abs(down)
    ax.bar(xi, total, color=col, edgecolor="none")

    ax.text(
        xi, total + total * 0.03,
        lbl,
        ha="center", va="bottom",
        fontsize=7, rotation=90
    )

ax.axhline(0, color="black", linewidth=1)
ax.set_xlim(-0.5, len(labels) - 0.5)
ax.set_ylabel("Number of DE features")

plt.tight_layout()
#plt.savefig(
#    sc.settings.figdir / "tf_ko_panel_de_results_summary_barplot_top5.pdf",
#    dpi=300
#)
plt.grid(False)
plt.show()

In [ ]:
# Union of top 5 per group
top5_per_group = (
    summary_df
    .assign(rank=lambda d: d.groupby("group")["total"].rank(method="first", ascending=False))
    .query("rank <= 5")
    .reset_index(drop=True)
)
union_top5_labels = list(set(top5_per_group["condition"].tolist()))
# Sort the union_top5_labels sum by group order

In [ ]:
n_de_df = summary_df.pivot_table(
    index="condition",
    columns="group",
    values="total",
    fill_value=0
)[group_order].loc[union_top5_labels]

In [ ]:
# Sort rows from highest to lowest total DE genes
n_de_df["sum"] = n_de_df.sum(axis=1)
n_de_df = n_de_df.sort_values("sum", ascending=False).drop(columns=["sum"])

In [ ]:
import matplotlib.pyplot as plt
import marsilea as ma
import marsilea.plotter as mp

# Copy data and mask zeros
data_masked = n_de_df.mask(n_de_df == 0)

# Build colormap where masked values appear as gray
cmap = plt.cm.RdPu.copy()
cmap.set_bad("lightgray")   # zeros → gray

# Plot
h = ma.Heatmap(data_masked, cmap=cmap)
h.add_right(mp.Labels(n_de_df.index))
h.add_bottom(mp.Labels(n_de_df.columns))
h.add_legends()
h.render()
plt.savefig(
    sc.settings.figdir / "tf_ko_panel_de_results_heatmap_top5_union.pdf",
    dpi=300
)